In [4]:
import copy
import numpy as np 
import matplotlib.pyplot as plt
from mnist.data_utils import *

In [2]:
# ----- Utils -----

In [ ]:
def sigmoid(z):
    
    return 1/(1+np.exp(-z))


def softmax(X):
    
    # logit = np.exp(x)이 아닌 이유
    # : 만약 지나치게 큰 값이 원소로 들어가게 된다면, 값이 너무 커서 연산이 되지 않는 오버플로 문제를 일으킬 위험이 있음
    # softmax에 들어가는 원소들에 대하여, 그 원소들의 최댓값을 빼는 것으로 해결 가능!
    logit = np.exp(X-np.amax(X, axis=1, keepdims=True))
    numer = logit
    denom = np.sum(logit, axis=1, keepdims=True)
    
    return numer/denom


def load_batch(X, Y, batch_size, shuffle=True):

    if shuffle:
        permutation = np.random.permutation(X.shape[0])
        X = X[permutation, :]
        Y = Y[permutation, :]
    
    num_steps = int(X.shape[0])//batch_size
    step = 0
    
    while step<num_steps:
        X_batch = X[batch_size*step:batch_size*(step+1)]
        Y_batch = Y[batch_size*step:batch_size*(step+1)]
        step+=1
        yield X_batch, Y_batch

In [3]:
# ----- 2-Layer Neural Network -----

In [ ]:
class TwoLayerNN:

    def __init__(self, input_dim, num_hiddens, num_classes):
        
        self.input_dim = input_dim
        self.num_hiddens = num_hiddens
        self.num_classes = num_classes
        self.params = self.initialize_parameters(input_dim, num_hiddens, num_classes)

    def initialize_parameters(self, input_dim, num_hiddens, num_classes):
        """
        initializes parameters with Xavier Initialization.
        - refer to https://paperswithcode.com/method/xavier-initialization
        
        Inputs
        - input_dim
        - num_hiddens
        - num_classes
        
        Returns
        - params: a dictionary with the initialized parameters.
        """
        params = {}
        params["W1"] = np.random.randn(input_dim, num_hiddens)/ np.sqrt(input_dim) 
        params["b1"] = np.zeros((num_hiddens, 1))
        params["W2"] = np.random.randn(num_hiddens, num_classes)/ np.sqrt(num_hiddens) 
        params["b2"] = np.zeros((num_classes, 1))

        return params

    
    def forward(self, X):
        """
        Define and perform the feed forward step of a two-layer neural network.
        Specifically, the network structue is given by

          y = softmax(sigmoid(X W1 + b1) W2 + b2)

        where X is the input matrix of shape (N, D), y is the class distribution matrix
        of shape (N, C), N is the number of examples (either the entire dataset or
        a mini-batch), D is the feature dimensionality, and C is the number of classes.

        Inputs
        - X: the input matrix of shape (N, D)

        Returns
        - y: the output of the model
        - ff_dict: a dictionary with all the fully connected units and activations.
        """
        params = self.params
        
        ff_dict = {}
        ff_dict['z1'] = np.dot(X, params["W1"]) + params["b1"]
        ff_dict['h1'] = sigmoid(ff_dict['z1'])
        ff_dict['z2'] = np.dot(ff_dict['h1'], params["W2"]) + params["b2"]
        ff_dict['h2'] = softmax(ff_dict['z2'])
        y = ff_dict['h2']
        
        return y, ff_dict

    
    def backward(self, X, Y, ff_dict):
        """
        Performs backpropagation over the two-layer neural network, and returns
        a dictionary of gradients of all model parameters.

        Inputs:
         - X: the input matrix of shape (B, D), where B is the number of examples
              in a mini-batch, D is the feature dimensionality.
         - Y: the matrix of one-hot encoded ground truth classes of shape (B, C),
              where B is the number of examples in a mini-batch, C is the number
              of classes.
         - ff_dict: the dictionary containing all the fully connected units and
              activations.

        Returns:
         - grads: a dictionary containing the gradients of corresponding weights and biases.
        """
        grads = {}
        z1 = ff_dict['z1']
        h1 = ff_dict['h1']
        z2 = ff_dict['z2']
        h2 = ff_dict['h2'].reshape(-1,1)
        dLdy = 2(h2-Y)
        dydz2 = np.diagflat(h2) - np.dot(h2, h2.T)
        dz2dW2 = h1
        grads["dW1"] = 2(h2-Y) *
        grads["db1"] =
        grads["dW2"] =
        grads["db2"] =
        
        return grads

    
# y = softmax(sigmoid(X*W1 + b1) W2 + b2)
#
# z1 = W1*X + b1
# h1 = sigmoid(z1)
# z2 = W2*h1 + b2
# h2 = softmax(W2*h1 + b2) = softmax(z2)
# y = h2
#
# L = (y-true_y)**2
# dLdy = 2(y-true_y)
# dLdW2 = dLdy * dydW2 = dLdy * dydz2 * dz2dW2 = 2(y-true_y) * so'(z2)' * h1
# dLdb2 = dLdy * dydb2 = dLdy * dydz2 * dz2db2 = 2(y-true_y) * so'(z2)'
# dLdW1 = dLdy * dydW1 = dLdy * dydz2 * dz2dh1 * dh1dz1 * dz1dW1 = 2(y-true_y) * so'(z2)' * W2 * h1(1-h1) * X
# dLdb1 = dLdy * dydb1 = dLdy * dydz2 * dz2dh1 * dh1dz1 * dz1db1 = 2(y-true_y) * so'(z2)' * W2 * h1(1-h1)
#
#
# if i == j:
#     self.gradient[i,j] = self.value[i] * (1-self.value[i])
# else:
#     self.gradient[i,j] = -self.value[i] * self.value[j]
    
    
    def compute_loss(self, Y, Y_hat):
        """
        Computes cross entropy loss
        """
        loss = -(1/Y.shape[0]) * np.sum(np.multiply(Y, np.log(Y_hat)))
        return loss

    
    def train(self, X, Y, X_val, Y_val, lr, n_epochs, batch_size, log_interval=1):
        """
        Runs mini-batch gradient descent.

        Do NOT Modify this method.

        Inputs
        - X
        - Y
        - X_val
        - Y_Val
        - lr
        - n_epochs
        - batch_size
        - log_interval
        """
        for epoch in range(n_epochs):
            for X_batch, Y_batch in load_batch(X, Y, batch_size):
                self.train_step(X_batch, Y_batch, batch_size, lr)
            if epoch % log_interval==0:
                Y_hat, ff_dict = self.forward(X)
                train_loss = self.compute_loss(Y, Y_hat)
                train_acc = self.evaluate(Y, Y_hat)
                Y_hat, ff_dict = self.forward(X_val)
                valid_loss = self.compute_loss(Y_val, Y_hat)
                valid_acc = self.evaluate(Y_val, Y_hat)
                print('epoch {:02} - train loss/acc: {:.3f} {:.3f}, valid loss/acc: {:.3f} {:.3f}'.\
                      format(epoch, train_loss, train_acc, valid_loss, valid_acc))

                
    def train_step(self, X_batch, Y_batch, batch_size, lr):
        """
        Updates the parameters using gradient descent.

        Inputs
        - X_batch
        - Y_batch
        - batch_size
        - lr
        """
        _, ff_dict = self.forward(X_batch)
        grads = self.backward(X_batch, Y_batch, ff_dict)
        self.params["W1"] -= lr * grads["dW1"]/batch_size
        self.params["b1"] -= lr * grads["db1"]/batch_size
        self.params["W2"] -= lr * grads["dW2"]/batch_size
        self.params["b2"] -= lr * grads["db2"]/batch_size

        
    def evaluate(self, Y, Y_hat):
        """
        Computes classification accuracy.

        Inputs
        - Y: A numpy array of shape (N, C) containing the softmax outputs,
             where C is the number of classes.
        - Y_hat: A numpy array of shape (N, C) containing the one-hot encoded labels,
             where C is the number of classes.

        Returns
            accuracy: the classification accuracy in float
        """        
        classes_pred = np.argmax(Y_hat, axis=1)
        classes_gt = np.argmax(Y, axis=1)
        accuracy = float(np.sum(classes_pred==classes_gt)) / Y.shape[0]
        return accuracy

In [1]:
# ----- Load MNIST -----

In [5]:
X_train, Y_train, X_test, Y_test = load_data()

idxs = np.arange(len(X_train))
np.random.shuffle(idxs)
split_idx = int(np.ceil(len(idxs)*0.8))
X_valid, Y_valid = X_train[idxs[split_idx:]], Y_train[idxs[split_idx:]]
X_train, Y_train = X_train[idxs[:split_idx]], Y_train[idxs[:split_idx]]
print()
print('Set validation data aside')
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', Y_train.shape)
print('Validation data shape: ', X_valid.shape)
print('Validation labels shape: ', Y_valid.shape)

MNIST data loaded:
Training data shape: (60000, 784)
Training labels shape: (60000, 10)
Test data shape: (10000, 784)
Test labels shape: (10000, 10)

Set validation data aside
Training data shape:  (48000, 784)
Training labels shape:  (48000, 10)
Validation data shape:  (12000, 784)
Validation labels shape:  (12000, 10)


In [7]:
# ----- Training & Evaluation -----

In [ ]:
# model instantiation
model = TwoLayerNN(input_dim=784, num_hiddens=64, num_classes=10)

In [ ]:
# train the model
lr, n_epochs, batch_size = 2.0, 20, 256
model.train(X_train, Y_train, X_valid, Y_valid, lr, n_epochs, batch_size)

In [ ]:
# evalute the model on test data
Y_hat, _ = model.forward(X_test)
test_loss = model.compute_loss(Y_test, Y_hat)
test_acc = model.evaluate(Y_test, Y_hat)
print("Final test loss = {:.3f}, acc = {:.3f}".format(test_loss, test_acc))

# Extra Credit (Optional)

In [ ]:
def initialize_parameters(self, input_dim, num_hiddens, num_classes):
    """
    initializes parameters with He Initialization.

    Question (e)
    - refer to https://paperswithcode.com/method/he-initialization for He initialization 
    
    Inputs
    - input_dim
    - num_hiddens
    - num_classes
    Returns
    - params: a dictionary with the initialized parameters.
    """
    params = {}
    
    return None

def forward_relu(self, X):
    """
    Defines and performs the feed forward step of a two-layer neural network.
    Specifically, the network structue is given by

        y = softmax(relu(X W1 + b1) W2 + b2)

    where X is the input matrix of shape (N, D), y is the class distribution matrix
    of shape (N, C), N is the number of examples (either the entire dataset or
    a mini-batch), D is the feature dimensionality, and C is the number of classes.

    Question (e)

    Inputs
        X: the input matrix of shape (N, D)

    Returns
        y: the output of the model
        ff_dict: a dictionary containing all the fully connected units and activations.
    """
    ff_dict = {}        
    
    return None

def backward_relu(self, X, Y, ff_dict):
    """
    Performs backpropagation over the two-layer neural network, and returns
    a dictionary of gradients of all model parameters.

    Question (e)

    Inputs:
        - X: the input matrix of shape (B, D), where B is the number of examples
            in a mini-batch, D is the feature dimensionality.
        - Y: the matrix of one-hot encoded ground truth classes of shape (B, C),
            where B is the number of examples in a mini-batch, C is the number
            of classes.
        - ff_dict: the dictionary containing all the fully connected units and
            activations.

    Returns:
        - grads: a dictionary containing the gradients of corresponding weights
            and biases.
    """
    grads = {}

    return None

TwoLayerNNRelu = copy.copy(TwoLayerNN)
TwoLayerNNRelu.initialize_parameters = initialize_parameters
TwoLayerNNRelu.forward = forward_relu
TwoLayerNNRelu.backward = backward_relu

In [ ]:
### 
# Question (e)
# Tune the hyperparameters with validation data,
# and print the results by running the lines below.
###

In [ ]:
# model instantiation
model_relu = TwoLayerNNRelu(input_dim=784, num_hiddens=64, num_classes=10)

In [ ]:
# train the model
lr, n_epochs, batch_size = 1.5, 20, 256
history = model_relu.train(X_train, Y_train, X_valid, Y_valid, lr, n_epochs, batch_size)

In [ ]:
Y_hat, _ = model_relu.forward(X_test)
test_loss = model_relu.compute_loss(Y_test, Y_hat)
test_acc = model_relu.evaluate(Y_test, Y_hat)
print("Final test loss = {:.3f}, acc = {:.3f}".format(test_loss, test_acc))